### To use LangChain RAG with Colab and Gemini API

    1. Loading the documents
    2. Splitting the documents
    3. Embedding the splits
    4. Storing the embeddings in the Vector Store
    5. Retrieval
    6. Augmented Generation

In [ ]:
!pip install langchain langchain-core langchain-community chromadb==0.5.3 langchain-chroma docx2txt pypdf
!pip install langchain_google_genai langchain-groq langchain-huggingface sentence-transformers

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.5/559.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Virtual_Characters.docx to Virtual_Characters.docx


In [ ]:
import os
#import dotenv
from pathlib import Path

from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders import (
    WebBaseLoader,
    PyPDFLoader,
    Docx2txtLoader,
)
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
#from langchain_openai import OpenAIEmbeddings, ChatOpenAI
#from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

#Kevin241130
from langchain_google_genai import ChatGoogleGenerativeAI #Google
from langchain_groq import ChatGroq #Groq
#from langchain.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

#dotenv.load_dotenv()
# 匯入套件和Google金鑰
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
# Load docs

doc_paths = [
    "test_rag.pdf",
    "Virtual_Characters.docx",
]

docs = []
for doc_file in doc_paths:
    file_path = Path(doc_file)

    try:
        if doc_file.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
        elif doc_file.endswith(".docx"):
            loader = Docx2txtLoader(file_path)
        elif doc_file.endswith(".txt") or doc_file.name.endswith(".md"):
            loader = TextLoader(file_path)
        else:
            print(f"Document type {doc_file.type} not supported.")
            continue

        docs.extend(loader.load())

    except Exception as e:
        print(f"Error loading document {doc_file.name}: {e}")


# Load URLs

url = "https://docs.streamlit.io/develop/quick-reference/release-notes"
try:
    loader = WebBaseLoader(url)
    docs.extend(loader.load())

except Exception as e:
    print(f"Error loading document from {url}: {e}")

In [ ]:
docs

[Document(metadata={'source': 'test_rag.pdf', 'page': 0}, page_content='My favorite food is margarita pizza. \nThere are 47588 bottles in the truck. '),
 Document(metadata={'source': 'Virtual_Characters.docx'}, page_content="Taylor Swift, a 28-year-old female researcher, stands out not only for her genius in the scientific community but also for her remarkable capacities in data analysis and cryptography. With an intelligence score of 90, a physical strength of 70, and a charm of 75, Taylor's talents extend beyond the conventional boundaries. Her ability to conduct complex experiments under extreme conditions has caused a significant stir in the academic world, propelling her to the forefront of her field. Her groundbreaking work has not only pushed the limits of academia but also attracted attention from governmental and private sectors.\n\nHowever, Taylor's true motives and goals remain shrouded in mystery. This enigmatic nature, coupled with her exceptional skill set, makes her adep

In [ ]:
# Split docs

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=1000,
)

document_chunks = text_splitter.split_documents(docs)

In [ ]:
#Kevin241201
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cpu'}
embedding = HuggingFaceEmbeddings(model_name=model_name,
                                  model_kwargs=model_kwargs)


# Tokenize and load the documents to the vector store
vector_db = Chroma.from_documents(
    documents=document_chunks,
    #embedding=OpenAIEmbeddings(),
    embedding=embedding, #Kevin241201
)

In [ ]:
# Retrieve

def _get_context_retriever_chain(vector_db, llm):
    retriever = vector_db.as_retriever()
    prompt = ChatPromptTemplate.from_messages([
        MessagesPlaceholder(variable_name="messages"),
        ("user", "{input}"),
        ("user", "Given the above conversation, generate a search query to look up in order to get inforamtion relevant to the conversation, focusing on the most recent messages."),
    ])
    retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

    return retriever_chain

In [ ]:
def get_conversational_rag_chain(llm):
    retriever_chain = _get_context_retriever_chain(vector_db, llm)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
        """You are a helpful assistant. You will have to answer to user's queries.
        You will have some context to help with your answers, but now always would be completely related or helpful.
        You can also use your knowledge to assist answering the user's queries.\n
        {context}"""),
        MessagesPlaceholder(variable_name="messages"),
        ("user", "{input}"),
    ])
    stuff_documents_chain = create_stuff_documents_chain(llm, prompt)

    return create_retrieval_chain(retriever_chain, stuff_documents_chain)

In [ ]:
# Augmented Generation

# llm_stream_openai = ChatOpenAI(
#     model="gpt-4o",  # Here you could use "o1-preview" or "o1-mini" if you already have access to them
#     temperature=0.3,
#     streaming=True,
# )

# llm_stream_anthropic = ChatAnthropic(
#     model="claude-3-5-sonnet-20240620",
#     temperature=0.3,
#     streaming=True,
# )

#Google
from langchain_google_genai import ChatGoogleGenerativeAI
os.environ["GOOGLE_API_KEY"] = os.getenv('GOOGLE_API_KEY')
llm_stream_google = ChatGoogleGenerativeAI(model="gemini-1.5-flash")


#llm_stream = llm_stream_openai  # Select between OpenAI and Anthropic models for the response
llm_stream = llm_stream_google



messages = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hi there! How can I assist you today?"},
    #{"role": "user", "content": "What is the latest version of Streamlit?"},
    {"role": "user", "content": "What is Taylor Swift's career and age?"},
]
messages = [HumanMessage(content=m["content"]) if m["role"] == "user" else AIMessage(content=m["content"]) for m in messages]
messages

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi there! How can I assist you today?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What is Taylor Swift's career and age?", additional_kwargs={}, response_metadata={})]

In [ ]:
conversation_rag_chain = get_conversational_rag_chain(llm_stream)
conversation_rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7aa22cc36da0>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['input', 'messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Anno

In [ ]:
response_message = "*(RAG Response)*\n"
for chunk in conversation_rag_chain.pick("answer").stream({"messages": messages[:-1], "input": messages[-1].content}):
    response_message += chunk
    #print(chunk, end="", flush=True)

print(response_message)
messages.append({"role": "assistant", "content": response_message})

Based on the provided text, Taylor Swift is a 28-year-old female researcher specializing in data analysis and cryptography.  Her career is in the scientific community, where she's known for her genius and ability to conduct complex experiments under extreme conditions.  Her work has gained attention from both academic and governmental/private sectors.
*(RAG Response)*
Based on the provided text, Taylor Swift is a 28-year-old female researcher specializing in data analysis and cryptography.  Her career is in the scientific community, where she's known for her genius and ability to conduct complex experiments under extreme conditions.  Her work has gained attention from both academic and governmental/private sectors.

